# 7B AWQ Quantization + Batched Trajectory Collection
Two parts:
1. AWQ-quantize Qwen2.5-7B-Instruct to 4-bit
2. Collect generation trajectories using the batched approach (BS=32, ~10 min for 500 trajs)

**Output**: Trajectories saved to Google Drive in batch files (`Trajs_7B/batch_*.pt`).

In [ ]:
# Install deps
!pip install -q autoawq transformers torch datasets accelerate

import os, glob, zipfile, time, gc, torch, re, json
import numpy as np
from awq import AutoAWQForCausalLM
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
from google.colab import drive, files
import threading

## Part 1: AWQ Quantization

In [ ]:
MODEL = "Qwen/Qwen2.5-7B-Instruct"
SAVE_DIR = "qwen7b_awq"
W_BIT = 4; GROUP_SIZE = 128; CALIB_SAMPLES = 128

print(f"Quantizing {MODEL} to {W_BIT}-bit AWQ...", flush=True)
model = AutoAWQForCausalLM.from_pretrained(MODEL, device_map='cuda')
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.1f}GB", flush=True)

In [ ]:
print("Quantizing...", flush=True)
model.quantize(
    tokenizer,
    quant_config={"zero_point": True, "q_group_size": GROUP_SIZE,
                  "w_bit": W_BIT, "version": "GEMM"},
    calib_data="pileval",
    max_calib_samples=CALIB_SAMPLES,
)
print("Done!", flush=True)

In [ ]:
DRIVE_MODEL_DIR = "/content/drive/MyDrive/TrimTab/qwen7b_awq/"
os.makedirs(SAVE_DIR, exist_ok=True)
model.save_quantized(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
!du -sh $SAVE_DIR

# Copy to Drive for persistent access
drive.mount('/content/drive', force_remount=False)
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)
!cp -r $SAVE_DIR/* "$DRIVE_MODEL_DIR"
print(f"Model saved to Drive: {DRIVE_MODEL_DIR}", flush=True)

# Also offer download (optional) — use if you want to download directly
ZIP_NAME = f"{SAVE_DIR}.zip"
with zipfile.ZipFile(ZIP_NAME, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(SAVE_DIR):
        for f in files:
            zf.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), SAVE_DIR))
print(f"Zipped: {os.path.getsize(ZIP_NAME)/1e6:.0f}MB (optional download below)")
files.download(ZIP_NAME)

## Part 2: Batched Trajectory Collection

Uses the quantized AWQ 7B model with BS=32, collecting trajectories to Google Drive.

In [ ]:
# Mount Drive
drive.mount('/content/drive')
DRIVE_DIR = "/content/drive/MyDrive/TrimTab/Trajs_7B/"
os.makedirs(DRIVE_DIR, exist_ok=True)

# Clean up AWQ load model to free VRAM
del model; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Load the quantized model via transformers
print("Loading AWQ 7B model...", flush=True); t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    SAVE_DIR, trust_remote_code=True, device_map='cuda', torch_dtype=torch.float16)
model.eval()
tok = AutoTokenizer.from_pretrained(SAVE_DIR, trust_remote_code=True)
tok.pad_token = tok.eos_token; tok.padding_side = "left"
print(f"Loaded in {time.time()-t0:.0f}s", flush=True)
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.1f}GB", flush=True)

In [ ]:
# Configuration
N_TRAJS = 500       # trajectories to collect
BS = 32             # batch size (fits in ~6GB VRAM for 7B AWQ)
MAX_NEW = 120       # max tokens per generation
N_LAYERS = 28       # 7B has 28 layers
D_MODEL = 3584      # 7B hidden dim
SAVE_EVERY = 50     # save a batch file every N trajectories

ds = load_dataset("openai/gsm8k", "main", split="test")
problems = [r for r in ds if len(r["question"]) > 50]

In [ ]:
# Trajectory collection with batched generation
buf_h, buf_v, batch_idx, count, t0 = [], [], 0, 0, time.time()

def extract_trajs(plen, gl, hs, n_layers):
    results = []
    for b in range(len(gl)):
        if gl[b] <= 1: results.append(None); continue
        hsv = [hs[li][b, plen:plen+gl[b]].float() for li in range(n_layers)]
        ha = torch.stack(hsv, dim=0)
        h_s = ha[:, :-1].permute(1, 0, 2).cpu().half()
        v_t = (ha[:, 1:] - ha[:, :-1]).permute(1, 0, 2).cpu().half()
        results.append((h_s, v_t))
    return results

for start in range(0, N_TRAJS, BS):
    batch = problems[start:start+BS]
    prompts = [f"Q: {p['question']}\nA:" for p in batch]
    enc = tok(prompts, return_tensors='pt', padding=True)
    iids = enc['input_ids'].to('cuda')
    am = enc['attention_mask'].to('cuda')
    plen = iids.shape[1]

    out = model.generate(iids, attention_mask=am, max_new_tokens=MAX_NEW,
                          do_sample=False, pad_token_id=tok.eos_token_id)
    gl = (out != tok.pad_token_id).sum(dim=1) - plen

    with torch.no_grad():
        fwd = model(out, use_cache=False, output_hidden_states=True)

    results = extract_trajs(plen, gl, fwd.hidden_states, N_LAYERS)
    for r in results:
        if r is not None:
            buf_h.append(r[0]); buf_v.append(r[1]); count += 1

    del out, fwd, iids, am, enc; gc.collect(); torch.cuda.empty_cache()

    if len(buf_h) >= SAVE_EVERY or count >= N_TRAJS:
        fname = f"{DRIVE_DIR}/batch_{batch_idx:04d}.pt"
        torch.save({'hidden_seqs': torch.cat(buf_h),
                    'velocity_targets': torch.cat(buf_v)}, fname)
        buf_h, buf_v = [], []; batch_idx += 1
        speed = count / (time.time() - t0) * 60
        print(f"  [{count}/{N_TRAJS}] {fname} ({speed:.0f} trajs/min)", flush=True)

open(f"{DRIVE_DIR}/done.flag", 'w').close()
print(f"DONE: {count} trajectories in {(time.time()-t0)/60:.1f} min", flush=True)

In [ ]:
# Verify
total = 0
for f in sorted(glob.glob(f"{DRIVE_DIR}/batch_*.pt")):
    d = torch.load(f, map_location='cpu')
    n = len(d['hidden_seqs'])
    print(f"  {os.path.basename(f)}: {n} trajs, {d['hidden_seqs'].shape}", flush=True)
    total += n; del d
print(f"\nTotal: {total} trajectories in {os.path.getsize(DRIVE_DIR)/1e9:.1f}GB", flush=True)
print(f"Saved to: {DRIVE_DIR}")